In [2]:
import asyncio
import os
import time
from typing import Literal, Optional
from dotenv import load_dotenv

from openai import AsyncOpenAI
import pandas as pd
from pydantic import BaseModel, Field
from tqdm.notebook import tqdm

from pprint import pprint

load_dotenv()

True

In [3]:
client = AsyncOpenAI(
    base_url=os.getenv("GENERATION_API_URL"),
    api_key=os.getenv("SCW_SECRET_KEY"),
)

In [4]:
df = pd.read_parquet('results_557k/sample_2000_extracted_policies.parquet')
gold = pd.read_json('gold_correlation_dataset_2026-03-11.json')

In [9]:
df

,openalex_id,chunk_idx,text,contains_policies,policies
0,W4306385914,0,explore a new and effective method for mariti...,False,[]
1,W3173622727,6,. Compositional analysis of lignocellulosic fe...,False,[]
2,W4238379039,1,"2012), shown in overview, however the\n384 pre...",False,[]
3,W4408679953,0,affecting breastfeeding practice among mother...,False,[]
4,W3092401747,1,for all ﬁve local sites in both August and No...,False,[]
...,...,...,...,...,...
1995,W4379144753,0,study aimed to develop a combustion technolog...,None,None
1996,W4389428043,2,the context can be changed only to a limited ...,True,[[SOCIAL] [INFORMATIONAL] [SCHOOL] Interventio...
1997,W4313209439,0,a distinct signature of contemporary global w...,False,[]
1998,W2408034890,0,Valplast® and DuraFlex™ are thermoplastic mat...,False,[]


In [10]:
tax = pd.read_parquet('results_557k/sample_2000_taxonomy_260303.parquet')

In [11]:
df = df.merge(tax.drop(columns='text'), on=['openalex_id', 'chunk_idx'], how='left')

In [39]:
chunks = set(gold.set_index(['openalex_id', 'chunk_idx']).index)
df = df[df.set_index(['openalex_id', 'chunk_idx']).index.isin(chunks)]

In [44]:
pdf = df[df.contains_policies==True].explode('policies').reset_index(drop=True)
pdf = pdf[pdf.policies.notna()].reset_index(drop=True)
len(pdf)

364

In [108]:
system_prompt = """
Your job is to precisely extract policy information from scientific texts, more specifically the impact direction of a policy along given impact dimensions.
Direction can be:
- positive if the text clearly suggests a beneficial impact of the policy on the dimension
- negative if the text clearly suggests a detrimental impact of the policy on the dimension
- neutral if the text suggests no significant or mixed impact of the policy on the dimension
- unknown if either the impact dimension is irrelevant to the text or if its direction cannot be inferred.

For planetary boundaries, positive means that the policy helps to stay within the boundaries (reducing climate change is positive), while negative means that it contributes to exceeding them.
For natural ressources, positive means that the policy helps to preserve them (reducing consumption), while negative means that it contributes to their depletion.

## Instructions
1. Read the text and the policy carefully.
2. For each requested dimension, determine the direction of impact based solely on explicit statements in the text.
3. Assign one of the four directions: **positive**, **negative**, **neutral**, or **unknown**.
4. Return all requested dimensions — do not skip any.
5. Use **unknown** when the text provides no relevant information; use **neutral** only when the text explicitly states no significant or mixed impact.
"""

In [ ]:
system_prompt = """
Your job is to precisely extract policy information from scientific texts, more specifically the impact direction of a policy along given impact dimensions.
Direction can be:
- positive if the text mentions a beneficial impact of the policy on the dimension
- negative if the text mentions a detrimental impact of the policy on the dimension
- neutral if the text clearly reports that there is no significant or mixed impact of the policy on the dimension
- unknown if either the impact dimension is irrelevant to the text/policy or if the text gives no information on its direction.

## Guidelines per dimension

| Category | Dimension | Positive Impact | Negative Impact |
|---|---|---|---|
| Human Needs | Nutrition | Improves food security, dietary quality, or access to sufficient nutrition. | Reduces food access, worsens hunger, or degrades nutritional quality. |
| Human Needs | Shelter and Living Conditions | Improves housing quality, safety, or access to adequate shelter. | Worsens housing conditions, increases homelessness, or degrades living environments. |
| Human Needs | Hygiene | Improves access to clean water, sanitation, or hygiene practices. | Reduces sanitation access or increases exposure to unhygienic conditions. |
| Human Needs | Clothing | Improves access to adequate, affordable, or appropriate clothing. | Reduces clothing access or affordability, especially for vulnerable groups. |
| Human Needs | Healthcare | Expands access to healthcare services or improves health outcomes. | Reduces healthcare access, quality, or affordability. |
| Human Needs | Education | Increases access to quality education or improves learning outcomes. | Reduces educational access, quality, or equity. |
| Human Needs | Communication and Information | Improves access to information, media, or communication tools. | Restricts information access, digital connectivity, or freedom of expression. |
| Human Needs | Mobility | Improves access to safe, affordable, or sustainable transportation. | Restricts movement, worsens transportation access, or increases travel-related inequity. |
| Natural Resources | Freshwater | Reduces freshwater consumption, improves water quality, or replenishes water sources. | Depletes freshwater reserves, increases pollution, or reduces water availability. |
| Natural Resources | Marine Resources | Protects marine ecosystems, reduces overfishing, or improves ocean health. | Depletes fish stocks, damages marine habitats, or increases ocean pollution. |
| Natural Resources | Wetlands | Preserves or restores wetland ecosystems and their biodiversity. | Drains, degrades, or destroys wetland areas. |
| Natural Resources | Metals and Ores | Reduces extraction, promotes recycling, or improves mining sustainability. | Increases extraction rates, waste, or environmental damage from mining. |
| Natural Resources | Non-Metallic Minerals | Reduces quarrying impacts or promotes more efficient use of minerals. | Increases extraction and associated land degradation or waste. |
| Natural Resources | Fossil Fuels | Reduces fossil fuel extraction or consumption, accelerating energy transition. | Increases fossil fuel use, dependence, or extraction activities. |
| Natural Resources | Agricultural Land | Improves land use efficiency, soil health, or sustainable farming practices. | Degrades soil quality, reduces arable land, or promotes unsustainable farming. |
| Natural Resources | Forests | Increases forest cover, prevents deforestation, or promotes reforestation. | Contributes to deforestation, habitat loss, or forest degradation. |
| Natural Resources | Urban Land | Promotes sustainable urban planning, green spaces, or efficient land use. | Increases urban sprawl, impervious surfaces, or inefficient land consumption. |
| Natural Resources | Biomass | Promotes sustainable biomass use without depleting natural ecosystems. | Overexploits biological resources or disrupts natural biomass cycles. |
| Wellbeing | Housing | Improves housing affordability, quality, or security of tenure. | Reduces housing affordability, increases overcrowding, or worsens living conditions. |
| Wellbeing | Jobs | Creates quality employment, reduces unemployment, or improves working conditions. | Destroys jobs, worsens labor conditions, or increases precarious employment. |
| Wellbeing | Education | Enhances access to or quality of education and lifelong learning. | Reduces educational opportunities or widens skill/knowledge gaps. |
| Wellbeing | Civic Engagement | Strengthens democratic participation, civic life, or community involvement. | Suppresses civic participation, political voice, or community engagement. |
| Wellbeing | Life Satisfaction | Increases overall subjective well-being, happiness, or sense of purpose. | Decreases people's sense of fulfillment, happiness, or life quality. |
| Wellbeing | Work-Life Balance | Reduces overwork, promotes flexibility, or supports personal time. | Increases work demands, reduces personal time, or worsens stress levels. |
| Wellbeing | Income | Raises incomes, reduces poverty, or improves financial security. | Reduces incomes, increases poverty, or worsens financial vulnerability. |
| Wellbeing | Community | Strengthens social bonds, trust, or community cohesion. | Weakens social ties, increases isolation, or fragments communities. |
| Wellbeing | Environment | Improves local environmental quality (air, water, green spaces). | Degrades local environmental conditions affecting people's well-being. |
| Wellbeing | Health | Improves physical or mental health outcomes for individuals or populations. | Worsens health outcomes, increases disease burden, or reduces life expectancy. |
| Wellbeing | Safety | Reduces crime, accidents, or exposure to hazards, increasing personal security. | Increases risks of harm, violence, or unsafe living/working conditions. |
| Justice Considerations | Distributional | Ensures benefits and burdens are shared fairly across society. | Concentrates benefits among the few or disproportionately burdens vulnerable groups. |
| Justice Considerations | Procedural | Ensures fair, transparent, and inclusive decision-making processes. | Excludes affected groups from decisions or lacks transparency and accountability. |
| Justice Considerations | Corrective | Addresses past harms, provides remediation, or compensates affected parties. | Fails to address historical injustices or worsens existing grievances. |
| Justice Considerations | Recognitional | Acknowledges and respects the identities, rights, and needs of all groups. | Ignores or marginalizes the voices, cultures, or identities of specific groups. |
| Justice Considerations | Transitional | Supports just transition for those negatively affected by systemic changes. | Leaves behind vulnerable groups during periods of economic or environmental transition. |
| Planetary Boundaries | Land-System Change | Reduces land conversion, protects natural habitats, or restores degraded land. | Accelerates conversion of natural land to agriculture, urban, or industrial use. |
| Planetary Boundaries | Climate Change | Reduces GHG/CO2 emissions or enhances carbon sequestration. | Increases greenhouse gas emissions, accelerating global warming. |
| Planetary Boundaries | Biosphere Integrity | Protects biodiversity, ecosystems, or prevents species extinction. | Accelerates biodiversity loss, species extinction, or ecosystem collapse. |
| Planetary Boundaries | Biogeochemical Flows | Reduces excess nitrogen/phosphorus flows into ecosystems. | Increases nutrient pollution, causing eutrophication or ecosystem disruption. |
| Planetary Boundaries | Ocean Acidification | Reduces CO2 emissions that cause ocean acidification. | Increases CO2 levels, worsening ocean acidification and harming marine life. |
| Planetary Boundaries | Freshwater Use | Reduces freshwater consumption or improves efficiency of water use. | Increases freshwater withdrawal beyond sustainable limits. |
| Planetary Boundaries | Atmospheric Aerosol Loading | Reduces air pollution particles that affect climate and human health. | Increases atmospheric aerosol emissions, disrupting weather and health. |
| Planetary Boundaries | Ozone Depletion | Reduces emissions of ozone-depleting substances. | Releases substances that break down the stratospheric ozone layer. |
| Planetary Boundaries | Introduction of Novel Entities | Limits release of new pollutants, chemicals, or materials into the environment. | Introduces new synthetic chemicals, plastics, or other novel agents into ecosystems. |
| Equity | Affordability | Makes goods, services, or opportunities more affordable for all, especially low-income groups. | Increases costs or financial barriers, making necessities less accessible. |
| Equity | Accessibility | Removes physical, digital, or social barriers to goods, services, or opportunities. | Creates or reinforces barriers that prevent people from accessing what they need. |
| Equity | Availability | Ensures sufficient supply of goods, services, or resources for all who need them. | Reduces supply or concentrates resources, leaving some groups without access. |

## Instructions
1. Read the text and the policy carefully.
2. For each requested dimension, determine the direction of impact based solely on explicit statements in the text.
3. Assign one of the four directions: **positive**, **negative**, **neutral**, or **unknown**.
4. Return all requested dimensions — do not skip any.
5. Use **unknown** when the text provides no relevant information; use **neutral** only when the text explicitly states no significant or mixed impact.
"""

In [109]:
impact_cols = ['justice_consideration_pred', 'planetary_boundaries_pred', 'wellbeing_pred', 'human_needs_pred', 'natural_ressource_pred']

def build_prompt(row):
    prompt = f"""<text>
{row.text}
</text>

<policy>
{row.policies.split(']')[-1].strip()}
</policy>

Extract the direction of impact along the following dimensions:
"""

    labels = []
    for col in impact_cols:
        if len(row[col]) > 0:
            prompt += f"- {col.replace('_pred', '').replace('_', ' ').title()}:\n"
            for label in row[col]:
                labels += [label]
                prompt += f"  - {label}\n\n"

    prompt += "YOU MUST refer the the second-level labels in your answer: " + ", ".join(labels)

    return prompt, labels


In [110]:
print(build_prompt(pdf.iloc[0])[0])

<text>
 considered relevant in the very early phases of deployment, other measures that encompass flexibility should be applied instead.
5.5 Regulation: Dissecting an umbrella-category
Throughout the reviewed literature, regulation is a recurring category [29,37,39,44–49,51,52,54,57,58]. In this study it proved relevant to subdivide regulation into separate issues to increase detail in the taxonomy.
5.6 Common solutions
Many of the barriers identified can be perceived as policy and regulatory risks. Especially barriers related to investment, but also bounded rationality and acceptance. Long-term policies and targets can mitigate those risks. In addition to the solutions in 3.2, is research and development [7,29,86]. Another option is direct
Page 22 of 31 regulation applying command and control solutions by government fiat. E.g. in the case of Denmark, where
DE must prove a >1 500 DKK/year/consumer saving, to be allowed to choose a biomass boiler over PTH
[131]. Common for all solutions

In [111]:
model_name = "mistral-small-3.2-24b-instruct-2506"
#model_name = "qwen/qwen3-235b-a22b-instruct-2507"

In [112]:
async def extract_impacts(
    row, model_name: str, client: AsyncOpenAI = client
):
    try:
        prompt, labels = build_prompt(row)
        
        class ImpactExtractionResponse(BaseModel):
            impacts: list[tuple[Literal[*labels], Literal['positive', 'negative', 'neutral', 'unknown']]] = Field(..., description="List of tuples (dimension, direction)")

        response = await client.chat.completions.create(
            model=model_name,
            messages=[
                {"role": "system", "content": system_prompt.strip()},
                {"role": "user", "content": prompt.strip()},
            ],
            temperature=0,
            max_tokens=512,
            response_format= {
                    "type": "json_schema",
                    "json_schema": {
                        "name": ImpactExtractionResponse.__name__,
                        "schema": ImpactExtractionResponse.model_json_schema(),
                    },
                },
            timeout=60,
            stream=False,
        )
        return ImpactExtractionResponse.model_validate_json(response.choices[0].message.content)
    except Exception as e:
        print('Error:', e)
        return f"Error: {e}"


async def batch_extract_impacts(
    batch: pd.DataFrame, model_name: str, client: AsyncOpenAI = client
):
    results = await asyncio.gather(*[extract_impacts(row, model_name, client) for _, row in batch.iterrows()])
    return results

In [113]:
i = 100
print(pdf.iloc[i].policies)
print('---')
print(pdf.iloc[i].text)

[INDUSTRY] [PARTICIPATORY] [COMPANIES] Involvement of stakeholders, particularly environmental authorities, in adopting measures to reduce the carbon footprint
---
 An innovation funnel is a tool used to transform an idea from the beginning of the creation process into reality by eliminating nonviable contributions along the way. The graphic form of the tool is a converging funnel which starts with the collection of all the inputs in order to investigate the feasible ideas, selecting the best of them and developing them to the point to deliver the ﬁnal product [33]. In our
12 of 18 study, a determination of CO2 reduction was not targeted, as was the case of [30], but the general list of development algorithms and tools can support companies in making their own assessments in terms of the effectiveness of employing these approaches to their own speciﬁc sub-sector of manufacturing.
As seen in Figure 7, the most frequently used technique is design for six sigma with the two variants of it

In [114]:
res = await extract_impacts(pdf.iloc[i], model_name, client)
res

ImpactExtractionResponse(impacts=[('biosphere_integrity', 'unknown'), ('climate_change', 'positive'), ('environment', 'positive'), ('income', 'unknown'), ('jobs', 'unknown')])

In [115]:
res = await batch_extract_impacts(pdf.iloc[20:30], model_name, client)
res

[ImpactExtractionResponse(impacts=[('distributional', 'unknown'), ('climate_change', 'positive'), ('environment', 'positive')]),
 ImpactExtractionResponse(impacts=[('distributional', 'unknown'), ('climate_change', 'positive'), ('environment', 'positive')]),
 ImpactExtractionResponse(impacts=[('distributional', 'unknown'), ('climate_change', 'positive'), ('environment', 'positive')]),
 ImpactExtractionResponse(impacts=[('environment', 'unknown'), ('jobs', 'unknown')]),
 ImpactExtractionResponse(impacts=[('environment', 'unknown'), ('jobs', 'unknown')]),
 ImpactExtractionResponse(impacts=[('environment', 'unknown'), ('jobs', 'unknown')]),
 ImpactExtractionResponse(impacts=[('environment', 'unknown'), ('jobs', 'unknown')]),
 ImpactExtractionResponse(impacts=[('environment', 'unknown'), ('jobs', 'unknown')]),
 ImpactExtractionResponse(impacts=[('environment', 'unknown'), ('jobs', 'unknown')]),
 ImpactExtractionResponse(impacts=[('environment', 'unknown'), ('jobs', 'unknown')])]

In [116]:
# process by batch
batch_size = 40
rpm_quota = 600
wait_time = 60 / (rpm_quota / batch_size)
results = []
for i in tqdm(range(0, len(pdf), batch_size)):
    batch = pdf.iloc[i:i+batch_size]
    preds = await batch_extract_impacts(batch, model_name, client)
    results.extend(preds)
    time.sleep(0.1)

  0%|          | 0/10 [00:00<?, ?it/s]

Error: 1 validation error for ImpactExtractionResponse
  Invalid JSON: EOF while parsing a list at line 3 column 506 [type=json_invalid, input_value='{\n  "impacts": [\n   \t...t\t\t\t\t\t\t\t\t\t\t\t', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/json_invalid
Error: 1 validation error for ImpactExtractionResponse
  Invalid JSON: EOF while parsing a list at line 3 column 506 [type=json_invalid, input_value='{\n  "impacts": [\n   \t...t\t\t\t\t\t\t\t\t\t\t\t', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/json_invalid
Error: 1 validation error for ImpactExtractionResponse
  Invalid JSON: EOF while parsing a list at line 3 column 506 [type=json_invalid, input_value='{\n  "impacts": [\n   \t...t\t\t\t\t\t\t\t\t\t\t\t', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/json_invalid
Error: 1 validation error for ImpactExtractionResponse
  Invalid JSON: EOF while parsing a list a

In [117]:
s = 0
for r in results:
    if isinstance(r, str) and r.startswith("Error:"):
        s += 1
print(f"{s} errors out of {len(results)} samples")

7 errors out of 364 samples


In [118]:
pdf['impacts_directions'] = [r.impacts if not isinstance(r, str) else [("error", "error")] for r in results]

In [38]:
pdf.to_parquet('results_557k/sample_2000_policies_impact_directions_2026-03-04_v2.parquet')

## Evaluation

In [119]:
rdf = pdf.explode('impacts_directions')[['openalex_id', 'chunk_idx', 'text', 'policies', 'impacts_directions']]

In [120]:
rdf['impact'], rdf['direction'] = zip(*rdf['impacts_directions'])

In [121]:
rdf = rdf.drop(columns='impacts_directions')
rdf = rdf.rename(columns={'policies': 'policy'})
rdf.head()

,openalex_id,chunk_idx,text,policy,impact,direction
0,W3107741367,8,considered relevant in the very early phases ...,[ENERGY] [REGULATORY] [NATIONAL] DE must prove...,procedural,neutral
0,W3107741367,8,considered relevant in the very early phases ...,[ENERGY] [REGULATORY] [NATIONAL] DE must prove...,climate_change,neutral
0,W3107741367,8,considered relevant in the very early phases ...,[ENERGY] [REGULATORY] [NATIONAL] DE must prove...,jobs,unknown
1,W3107741367,8,considered relevant in the very early phases ...,[ENERGY] [REGULATORY] [NATIONAL] Externally im...,procedural,unknown
1,W3107741367,8,considered relevant in the very early phases ...,[ENERGY] [REGULATORY] [NATIONAL] Externally im...,climate_change,unknown


In [122]:
jdf = gold.set_index(['openalex_id', 'chunk_idx', 'policy', 'impact']).join(rdf.set_index(['openalex_id', 'chunk_idx', 'policy', 'impact']), how='outer').reset_index()

In [123]:
jdf = jdf[jdf.direction != 'error']

In [124]:
jdf.isna().sum()

openalex_id            0
chunk_idx              0
policy                 0
impact                 0
row_id                 0
original_direction     0
reviewed_direction     0
reviewed_at            0
text                  21
direction             21
dtype: int64

In [125]:
jdf = jdf.dropna()

In [105]:
from sklearn.metrics import classification_report

In [107]:
labels = jdf['reviewed_direction']
print(classification_report(labels, jdf['direction']))
print(classification_report(labels, jdf['original_direction']))

              precision    recall  f1-score   support

    negative       0.27      0.74      0.39        39
     neutral       0.15      0.29      0.20        14
    positive       0.74      0.84      0.79       674
     unknown       0.91      0.75      0.83      1008

    accuracy                           0.78      1735
   macro avg       0.52      0.66      0.55      1735
weighted avg       0.82      0.78      0.80      1735

              precision    recall  f1-score   support

                   0.00      0.00      0.00         0
    negative       0.47      0.72      0.57        39
     neutral       0.60      0.64      0.62        14
    positive       0.91      0.77      0.83       674
     unknown       0.97      0.89      0.93      1008

    accuracy                           0.84      1735
   macro avg       0.59      0.60      0.59      1735
weighted avg       0.93      0.84      0.88      1735



/home/francois/projects/13_democratiser_sobriete/policy_analysis/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/francois/projects/13_democratiser_sobriete/policy_analysis/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/francois/projects/13_democratiser_sobriete/policy_analysis/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_divi

In [126]:
labels = jdf['reviewed_direction']
print(classification_report(labels, jdf['direction']))
print(classification_report(labels, jdf['original_direction']))

              precision    recall  f1-score   support

    negative       0.32      0.59      0.41        39
     neutral       0.04      0.64      0.08        14
    positive       0.79      0.85      0.82       672
     unknown       0.94      0.69      0.80      1008

    accuracy                           0.75      1733
   macro avg       0.52      0.69      0.53      1733
weighted avg       0.86      0.75      0.79      1733

              precision    recall  f1-score   support

                   0.00      0.00      0.00         0
    negative       0.47      0.72      0.57        39
     neutral       0.60      0.64      0.62        14
    positive       0.91      0.77      0.83       672
     unknown       0.97      0.89      0.93      1008

    accuracy                           0.84      1733
   macro avg       0.59      0.61      0.59      1733
weighted avg       0.93      0.84      0.88      1733



/home/francois/projects/13_democratiser_sobriete/policy_analysis/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/francois/projects/13_democratiser_sobriete/policy_analysis/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/francois/projects/13_democratiser_sobriete/policy_analysis/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_divi

## Save gold dataset

In [130]:
from enum import Enum

In [131]:
class Need(str, Enum):
    nutrition = "Nutrition"
    shelter_and_living_conditions = "Shelter and living conditions"
    hygiene = "Hygiene"
    clothing = "Clothing"
    healthcare = "Healthcare"
    education = "Education"
    communication_and_information = "Communication and information"
    mobility = "Mobility"


class Resource(str, Enum):
    freshwater = "Freshwater"
    marine_resources = "Marine Resources"
    wetlands = "Wetlands"
    metals_and_ores = "Metals and Ores"
    non_metallic_minerals = "Non-Metallic Minerals"
    fossil_fuels = "Fossil Fuels"
    agricultural_land = "Agricultural Land"
    forests = "Forests"
    urban_land = "Urban Land"
    biomass = "Biomass"


class Wellbeing(str, Enum):
    housing = "Housing"
    jobs = "Jobs"
    education = "Education"
    civic_engagement = "Civic Engagement"
    life_satisfaction = "Life Satisfaction"
    work_life_balance = "Work-Life Balance"
    income = "Income"
    community = "Community"
    environment = "Environment"
    health = "Health"
    safety = "Safety"


class Justice(str, Enum):
    distributional = "Distributional"
    procedural = "Procedural"
    corrective = "Corrective"
    recognitional = "Recognitional"
    transitional = "Transitional"


class PlanetaryBoundary(str, Enum):
    land_system_change = "Land-System Change"
    climate_change = "Climate Change"
    biosphere_integrity = "Biosphere Integrity"
    biogeochemical_flows = "Biogeochemical Flows"
    ocean_acidification = "Ocean Acidification"
    freshwater_use = "Freshwater Use"
    atmospheric_aerosol_loading = "Atmospheric Aerosol Loading"
    ozone_depletion = "Ozone Depletion"
    introduction_of_novel_entities = "Introduction of Novel Entities"


In [132]:
fdf = jdf[['openalex_id', 'chunk_idx', 'text', 'policy', 'impact', 'reviewed_direction']]

In [137]:
def find_enum(value: str):
    for enum_class in [Need, Resource, Wellbeing, Justice, PlanetaryBoundary]:
        for item in enum_class:
            if item.name == value:
                return enum_class.__name__
    return None

In [139]:
fdf['impact_category'] = fdf['impact'].apply(find_enum)

In [149]:
fdf = fdf[['openalex_id', 'chunk_idx', 'text', 'policy', 'impact_category', 'impact', 'reviewed_direction']]

In [151]:
fdf = fdf.rename(columns={'reviewed_direction': 'direction'})

In [154]:
fdf.direction.value_counts()

direction
unknown     1008
positive     672
negative      39
neutral       14
Name: count, dtype: int64

In [155]:
fdf.to_parquet('gold_impact_direction.parquet')